In [ ]:
import os
import sys
import collections
import collections.abc

collections.Iterable = collections.abc.Iterable

import numpy as np
import pandas as pd
import scipy.sparse

import torch
import scanpy as sc
import anndata as ad

import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('..'))
from src import plotting, stats
from src.augmentation import scvi_balancer, mixup_balancer

plotting.setup_publication_style()

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Active processing device: {device}")

# SPANN bug patching

In [ ]:
import os

spann_model_path = os.path.abspath('../external/SPANN/spann/model.py')

try:
    with open(spann_model_path, 'r') as file:
        filedata = file.read()

    # Define targets for patching
    target_1 = 'id_target_high_conf = id_target[high_conf_idx]'
    replace_1 = 'id_target_high_conf = id_target[high_conf_idx.cpu()]'

    target_2 = 'batch_coor_high_conf = batch_coor[high_conf_idx]'
    replace_2 = 'batch_coor_high_conf = batch_coor[high_conf_idx.cpu()]'

    changes_made = 0

    if target_1 in filedata:
        filedata = filedata.replace(target_1, replace_1)
        changes_made += 1
        print("Successfully patched tensor device mismatch at target 1.")

    if target_2 in filedata:
        filedata = filedata.replace(target_2, replace_2)
        changes_made += 1
        print("Successfully patched tensor device mismatch at target 2.")

    if changes_made > 0:
        with open(spann_model_path, 'w') as file:
            file.write(filedata)
        print(f"SPANN model.py updated ({changes_made} lines patched).")
    else:
        print("No patching required (SPANN model.py is already up to date).")

except FileNotFoundError:
    print("Error: SPANN module not found. Did you run 'git submodule update --init'?")
except Exception as e:
    print(f"Unexpected error during patching: {e}")

# Data loading

In [ ]:
path_rna = '/data/seqFISH_40/adata_rna.h5ad'
path_spatial = '/data/seqFISH_40/adata_seqfish_40.h5ad'

adata_rna = sc.read_h5ad(path_rna)g
adata_seqfish = sc.read_h5ad(path_spatial)

# Enforce float32 for model stability and memory efficiency
for name, adata in [("RNA", adata_rna), ("Spatial", adata_seqfish)]:
    if scipy.sparse.issparse(adata.X):
        adata.X.data = adata.X.data.astype(np.float32)
    else:
        adata.X = adata.X.astype(np.float32)
    print(f"{name} dataset loaded: {adata.n_obs} cells x {adata.n_vars} genes")

# Remove ground truth from spatial dataset to simulate real-world annotation
if 'cell_type' in adata_seqfish.obs.columns:
    adata_seqfish.obs = adata_seqfish.obs.drop('cell_type', axis=1)
    print("Dropped 'cell_type' from spatial dataset.")

# Imbalance Measurement

In [ ]:
from src import stats, plotting

print("Analyzing cell type distribution and imbalance metrics...")

adata_seqfish_raw = sc.read_h5ad(path_spatial)

rna_labels = adata_rna.obs['cell_type'].astype(str).values
rna_metrics = stats.calculate_imbalance_metrics(rna_labels)

spatial_labels = adata_seqfish_raw.obs['cell_type'].astype(str).values
spatial_metrics = stats.calculate_imbalance_metrics(spatial_labels)

print("\nSpatial Ground Truth Imbalance Metrics:")
print(f"Imbalance Ratio (IR) : {spatial_metrics['IR']:.2f}")
print(f"Gini Coefficient     : {spatial_metrics['Gini']:.4f}")

print("\nGenerating distribution plots...")
dataset_dict = {
    'scRNA Reference': adata_rna,
    'Spatial Ground Truth': adata_seqfish_raw
}

plotting.plot_celltype_distribution(
    adata_dict=dataset_dict, 
    label_col='cell_type', 
    filename='celltype_distribution_before_balancing'
)

# Cell type balancing

In [ ]:
AUGMENTATION_METHOD = "scVI"  # Choose between "scVI", "Mixup", or "None"

print(f"Executing {AUGMENTATION_METHOD} Augmentation")

if AUGMENTATION_METHOD == "scVI":
    # Using the standardized generative approach
    adata_rna_balanced = scvi_balancer.augment_data(
        adata=adata_rna,
        labels_key="cell_type",
        covariate_batch_key="batch",  # Change to None if batch correction is not needed
        layer_key=None,               # Set to "counts" if raw counts are in a specific layer
        max_epochs=5000
    )
elif AUGMENTATION_METHOD == "Mixup":
    # Using the stochastic interpolation approach
    adata_rna_balanced = mixup_balancer.augment_data(
        adata=adata_rna, 
        label_col='cell_type'
    )
else:
    # Baseline (No augmentation)
    print("Proceeding with imbalanced baseline dataset.")
    adata_rna_balanced = adata_rna.copy()

adata_rna_balanced.obs["cell_type"] = adata_rna_balanced.obs["cell_type"].astype(str)
print(f"Final balanced RNA dataset shape: {adata_rna_balanced.shape}")

# Preprocessing

In [ ]:
spann_module_path = os.path.abspath('../external/SPANN/spann')
if spann_module_path not in sys.path:
    sys.path.append(spann_module_path)

In [ ]:
from preprocess import anndata_preprocess, generate_dataloaders
from model import generate_ae_params, SPANN_model


adata_cm, adata_spa, adata_rna_spann = anndata_preprocess(
    adata_seqfish, 
    adata_rna_balanced, 
    spatial_labels=False
)

batch_size = 256

(source_sp_ds, target_sp_ds, 
 source_cm_dl, target_cm_dl, 
 test_source_cm_dl, test_target_cm_dl) = generate_dataloaders(
    adata_cm, 
    adata_spa, 
    adata_rna_spann, 
    batch_size=batch_size
)

# SPANN model nitialization and training

In [ ]:
enc, dec, x_dim, z_dim = generate_ae_params(adata_cm, adata_spa, adata_rna_spann)

class_num = len(adata_rna_spann.obs['cell_type'].unique())
spann = SPANN_model(
    x_dim=x_dim, 
    z_dim=z_dim, 
    enc=enc, 
    dec=dec, 
    class_num=class_num, 
    device=device
)

print("Starting SPANN training loop (this may take a while depending on GPU)...")

adata_source, adata_target, threshold_test = spann.train(
    source_cm_dl, target_cm_dl, 
    source_sp_ds, target_sp_ds, 
    adata_spa.obs[["X", "Y"]],
    test_source_cm_dl, test_target_cm_dl,
    np.array(adata_rna_spann.obs['labels']),
    np.unique(adata_rna_spann.obs['cell_type']),
    lr=2e-4, resolution=0.5, lambda_spa=0.001, lambda_cd=0.001, lambda_nb=10,
    maxiter=5000, miditer1=2000, miditer2=4000, miditer3=4000
)

print("Training phase completed.")

# Visualization

In [ ]:
adata_total = ad.concat(
    [adata_source, adata_target], 
    label="source", 
    keys=["scRNA", "Spatial"],
    index_unique="-"
)

sc.pp.neighbors(adata_total, use_rep="X")
sc.tl.umap(adata_total)

fig_source = sc.pl.umap(adata_total, color="source", show=False, return_fig=True)
plotting.save_figure(fig_source, 'umap_integration_source', format='pdf')
plt.show()

adata_rna_umap = adata_total[adata_total.obs['source'] == 'scRNA']
fig_rna = sc.pl.umap(adata_rna_umap, color="cell_type", show=False, return_fig=True)
plotting.save_figure(fig_rna, 'umap_rna_celltype', format='pdf')
plt.show()

adata_spa_umap = adata_total[adata_total.obs['source'] == 'Spatial']
fig_spa = sc.pl.umap(adata_spa_umap, color="pred_cls", show=False, return_fig=True)
plotting.save_figure(fig_spa, 'umap_spatial_predicted', format='pdf')
plt.show()

In [ ]:
fig_spatial, ax_spatial = plt.subplots(figsize=(8, 8))

sns.scatterplot(
    data=adata_target.obs, 
    x='X', 
    y='Y', 
    hue='pred_cls', 
    s=8,                 
    edgecolor='none',    
    ax=ax_spatial
)

ax_spatial.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., frameon=False)
ax_spatial.set_aspect('equal', 'datalim') # Ensure physical spatial coordinates are not distorted
ax_spatial.set_title("Spatial Map of Predicted Cell Types")

plotting.save_figure(fig_spatial, 'spatial_map_predictions', format='pdf')
plt.show()

print("Plotting Escore distribution...")

fig_dist, ax_dist = plt.subplots(figsize=(6, 4))
sns.histplot(
    adata_target.obs['Escore'], 
    bins=25, 
    kde=False, 
    color='gray',
    ax=ax_dist
)

ax_dist.axvline(threshold_test, color='red', linestyle='--', linewidth=1.5, label=f'Threshold: {threshold_test:.2f}')
ax_dist.set_ylim(0, 4000)
ax_dist.set_xlabel("Entropy Score (Escore)")
ax_dist.set_ylabel("Frequency")
ax_dist.set_title("Distribution of Prediction Uncertainty")
ax_dist.legend(frameon=False)

plotting.save_figure(fig_dist, 'escore_distribution', format='pdf')
plt.show()

# Evaluation

In [ ]:
from src import evaluation, plotting

common_cells = adata_target.obs.index.intersection(adata_seqfish.obs.index)
adata_true_raw = sc.read_h5ad('../data/raw/adata_seqfish_40.h5ad')

y_true = adata_true_raw.obs.loc[common_cells, 'cell_type'].astype(str).values
y_pred = adata_target.obs.loc[common_cells, 'pred_cls'].astype(str).values

assert len(y_true) == len(y_pred)
has_unknown = 'Unknown' in np.unique(y_pred)

if has_unknown:
    print("Evaluating open-set classification (novel discovery mode)...")
    
    KNOWN_TYPES = sorted(adata_rna.obs["cell_type"].unique().astype(str))
    metrics = evaluation.evaluate_novel_discovery(y_true, y_pred, KNOWN_TYPES)
    
    print("\nNovel Cell Discovery Metrics:")
    print(f"K-ACC (Known Accuracy)      : {metrics['K_ACC']:.4f} (n={metrics['n_known']})")
    print(f"U-ACC (Novel Discovery Rate): {metrics['U_ACC']:.4f} (n={metrics['n_novel']})")
    print(f"T-ACC (Total Accuracy)      : {metrics['T_ACC']:.4f}")
    print(f"H-Score (Harmonic Mean)     : {metrics['H_SCORE']:.4f}")
    
    # Academic observations instead of casual warnings
    if metrics['U_ACC'] < 0.1:
        print("Note: Low U-ACC indicates potential overconfidence in known class assignment. Consider tuning the Escore threshold.")
    elif metrics['K_ACC'] < 0.1:
        print("Note: Low K-ACC indicates conservative assignment behavior (excessive 'Unknown' predictions).")
        
    plotting.plot_novel_discovery_confusion_matrix(y_true, y_pred, KNOWN_TYPES, filename='confusion_matrix_novel')

else:
    print("Evaluating closed-set classification (standard mode)...")
    
    metrics = evaluation.evaluate_standard_classification(y_true, y_pred)
    
    print("\nStandard Classification Metrics:")
    print(f"Accuracy (ACC)               : {metrics['ACC']:.4f}")
    print(f"Normalized Mutual Info (NMI) : {metrics['NMI']:.4f}")
    print(f"Adjusted Rand Index (ARI)    : {metrics['ARI']:.4f}")
    
    plotting.plot_standard_confusion_matrix(y_true, y_pred, filename='confusion_matrix_standard')